# Notebook 015 — Cross-Matching AI Discoveries with Astronomical Catalogs

## Objective

The previous notebooks measured and clustered astronomical objects using
image-derived features.

In this notebook we begin teaching **Einstein**, the Cosmic Intelligence
Lab research assistant, how to connect those measurements with the
accumulated knowledge of modern astronomy.

The workflow is:

1. Convert image coordinates into celestial coordinates.
2. Cross-match each object with astronomical databases.
3. Retrieve published information.
4. Compare AI discoveries with established astronomical knowledge.
5. Identify the most scientifically interesting objects for future study.

---

## Scientific Motivation

Machine learning can identify statistically interesting objects within an
image, but astronomy is built upon decades of observations and published
research.

By combining AI-derived measurements with professional astronomical
catalogs, Einstein becomes more than an image-analysis tool—it becomes an
AI-assisted research companion capable of linking observations to the
broader astronomical literature.

---

## Einstein's Research Workflow

```
Hubble Image
      ↓
Source Detection
      ↓
Feature Measurements
      ↓
Machine Learning
      ↓
Object Coordinates
      ↓
Astronomical Databases
      ↓
Scientific Knowledge
      ↓
Research Journal
```

In [ ]:
# Cell 1a - Investigation target 

# TARGET_LABEL = 45

In [1]:
# Cell 2 - Import Libraries ===================================

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

import astropy.units as u

In [2]:
# Cell 3 - Load Data ===========================================

image_file = Path("../data/raw/mastDownload/HST/ib5x1fetq/ib5x1fetq_flt.fits")

with fits.open(image_file) as hdul:

    image = hdul[1].data

    header = hdul[1].header

catalog = Table.read("../data/catalogs/clean_source_catalog.ecsv")

print("="*60)
print("Data Loaded")
print("="*60)

print(f"Image size : {image.shape}")
print(f"Catalog    : {len(catalog)} objects")

Data Loaded
Image size : (1014, 1014)
Catalog    : 79 objects


In [3]:
# Cell 3a - recreate clustering =======================

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = np.column_stack([
    catalog["segment_flux"],
    catalog["area"],
    catalog["semimajor_axis"],
    catalog["semiminor_axis"],
    catalog["eccentricity"],
    catalog["orientation"]
])

X = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42)
catalog["cluster"] = kmeans.fit_predict(X)

In [4]:
# Cell 4 - Build World Coordinate System =======================

wcs = WCS(header)

print("="*60)
print("World Coordinate System")
print("="*60)

print(wcs)

World Coordinate System
WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN-SIP' 'DEC--TAN-SIP'
CUNIT : 'deg' 'deg'
CRVAL : 53.159744356671 -27.783652344669
CRPIX : 507.0 507.0
CD1_1 CD1_2  : 2.3560466981293e-05 2.6243677825723e-05
CD2_1 CD2_2  : 2.9301084913961e-05 -2.095385200011e-05
NAXIS : 1014  1014


In [5]:
# Cell 5 - Convert One Object to Sky Coordinates ===============

test = catalog[catalog["label"] == 45][0]

sky = wcs.pixel_to_world(
    test["x_centroid"],
    test["y_centroid"]
)

print("="*60)
print("Test Object")
print("="*60)

print(f"Label : {test['label']}")

print()

print(f"RA  : {sky.ra.to_string(unit=u.hour, precision=2)}")

print(f"Dec : {sky.dec.to_string(precision=2)}")

Test Object
Label : 45

RA  : 3h32m40.78s
Dec : -27d46m16.06s


In [6]:
# Cell 6 - Einstein Master Catalog =============================

ra_list = []
dec_list = []

for obj in catalog:

    sky = wcs.pixel_to_world(
        obj["x_centroid"],
        obj["y_centroid"]
    )

    ra_list.append(sky.ra.deg)
    dec_list.append(sky.dec.deg)

catalog["ra_deg"] = ra_list
catalog["dec_deg"] = dec_list

print("=" * 60)
print("Einstein Master Catalog")
print("=" * 60)

print(f"Objects : {len(catalog)}")

print("RA/Dec columns added.")

Einstein Master Catalog
Objects : 79
RA/Dec columns added.


In [7]:
# Cell 7 - Bright Objects with Coordinates =====================

bright_catalog = catalog[catalog["cluster"] == 1]

bright_catalog.sort("segment_flux", reverse=True)

bright_catalog[
    "label",
    "segment_flux",
    "ra_deg",
    "dec_deg"
]

label,segment_flux,ra_deg,dec_deg
int32,float64,float64,float64
61,1051.520471215248,53.15838296912401,-27.794983504696855
51,482.0046505331993,53.15547043087871,-27.791562298587333
79,243.46851658821106,53.16343740550713,-27.7996018086203
14,234.4003678560257,53.154971262745285,-27.76898975952854
77,205.649180829525,53.17253333961878,-27.788169390170232
21,164.6831734776497,53.142144323970186,-27.786782256275846
45,137.40018010139465,53.16993598680541,-27.771127189673585
41,124.95641833543777,53.16234307574722,-27.775132188321017


In [8]:
# Cell 8 - write & save Einstein Master Catalog

catalog.write(
    "../data/catalogs/einstein_master_catalog.ecsv",
    format="ascii.ecsv",
    overwrite=True
)

print("Einstein Master Catalog saved.")

Einstein Master Catalog saved.


In [9]:
print("Cluster populations:")
for i in range(3):
    print(f"Cluster {i}: {(catalog['cluster'] == i).sum()} objects")

Cluster populations:
Cluster 0: 40 objects
Cluster 1: 8 objects
Cluster 2: 31 objects


In [10]:
cluster_means = []

for i in range(3):
    mean_flux = np.mean(catalog["segment_flux"][catalog["cluster"] == i])
    cluster_means.append(mean_flux)
    print(f"Cluster {i}: mean flux = {mean_flux:.2f}")

Cluster 0: mean flux = 7.68
Cluster 1: mean flux = 330.51
Cluster 2: mean flux = 53.96


In [11]:
cluster_means = []

for i in range(3):
    mask = catalog["cluster"] == i
    mean_flux = np.mean(catalog["segment_flux"][mask])
    cluster_means.append(mean_flux)
    print(f"Cluster {i}:")
    print(f"  Objects   : {mask.sum()}")
    print(f"  Mean Flux : {mean_flux:.2f}")
    print()

Cluster 0:
  Objects   : 40
  Mean Flux : 7.68

Cluster 1:
  Objects   : 8
  Mean Flux : 330.51

Cluster 2:
  Objects   : 31
  Mean Flux : 53.96



In [12]:
# Compare the old bright-object labels with the new clustering

old_bright = {14, 21, 39, 41, 45, 49, 51, 52, 61, 77, 79}

new_bright = set(bright_catalog["label"])

print("Objects remaining in bright cluster:")
print(sorted(old_bright & new_bright))

print("\nObjects that left the bright cluster:")
print(sorted(old_bright - new_bright))

print("\nNew objects entering bright cluster:")
print(sorted(new_bright - old_bright))

Objects remaining in bright cluster:
[np.int32(14), np.int32(21), np.int32(41), np.int32(45), np.int32(51), np.int32(61), np.int32(77), np.int32(79)]

Objects that left the bright cluster:
[39, 49, 52]

New objects entering bright cluster:
[]


In [13]:
# Compare two borderline objects

for label in [21, 52]:
    obj = catalog[catalog["label"] == label][0]

    print("=" * 60)
    print(f"Label {label}")
    print("=" * 60)

    print(f"Cluster         : {obj['cluster']}")
    print(f"Flux            : {obj['segment_flux']:.2f}")
    print(f"Area            : {obj['area']}")
    print(f"Semi-major axis : {obj['semimajor_axis']:.2f}")
    print(f"Semi-minor axis : {obj['semiminor_axis']:.2f}")
    print(f"Eccentricity    : {obj['eccentricity']:.3f}")
    print(f"Orientation     : {obj['orientation']:.2f}")

Label 21
Cluster         : 1
Flux            : 164.68
Area            : 96.0
Semi-major axis : 3.05
Semi-minor axis : 1.72
Eccentricity    : 0.826
Orientation     : 352.21
Label 52
Cluster         : 2
Flux            : 168.57
Area            : 67.0
Semi-major axis : 2.75
Semi-minor axis : 1.05
Eccentricity    : 0.924
Orientation     : 45.54


In [14]:
# ============================================================
# Notebook Status
# ============================================================

from datetime import datetime

print("=" * 60)
print("Notebook Status")
print("=" * 60)

print("Status    : PASS")
print("Notebook  : 15_crossmatching_ai_discoveries.ipynb")
print("Completed :", datetime.now().strftime("%Y-%m-%d %H:%M"))

Notebook Status
Status    : PASS
Notebook  : 15_crossmatching_ai_discoveries.ipynb
Completed : 2026-07-17 13:14
